# Functions for cross tabulated format (CTF) 1

Created: 2026-11-8

The previous notebook in this series [Non-pivot xslx functions 3](non-pivot-xlsx-03.ipynb) worked through functionality for cross tabulated format sheets in the [December 2014 statistics for foreign residents in Japan](http://www.e-stat.go.jp/SG1/estat/List.do?lid=000001133760) using tree data structures for parsing the stub column and column headers. This notebook is for adding this new functionality to the `excel` module. 

## Install the necessary packages

In [ ]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install openpyxl
%pip install treelib

## Install the newest version of estatjp

Repeat during development to import revisions.

## Data source urls

Workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

In [1]:
url20120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104290&fileKind=4"
url20120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104291&fileKind=0" # column-oriented
url20120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104292&fileKind=4"
url20120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104293&fileKind=0" # column-oriented
url201203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104294&fileKind=4"
url201204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104295&fileKind=4"
url201205 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032104296&fileKind=4"

## Import the module and download a file

In [2]:
from estatjp.xlsx import xlsx
from pathlib import Path

theurl = url20120101
dataset = "201212"
ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw/isa") / dataset
if Path.exists(dataraw) == False:
    Path.mkdir(dataraw)
dataprocessed = ROOT_DIR / Path(".data/processed/isa") / dataset
if Path.exists(dataprocessed) == False:
    Path.mkdir(dataprocessed)
newdata = xlsx.download_to_folder(theurl, dataraw, overwrite=True, verbose=True)
print('source: ' + str(newdata))
if newdata != None:
    processeddata = dataprocessed / newdata.name
    processeddata = processeddata.with_suffix(".csv")
    print('target: ' + str(processeddata))

{'Content-Type': 'application/octet-stream', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Accept-Ranges': 'bytes', 'X-Content-Type-Options': 'nosniff', 'Strict-Transport-Security': 'max-age=0', 'Date': 'Tue, 10 Nov 2026 05:01:10 GMT', 'X-Zen-Fury': '131153a915f438faeefaf5e82b50435c2bc2ecfa', 'X-Frame-Options': 'SAMEORIGIN', 'X-XSS-Protection': '1; mode=block', 'Content-Encoding': 'gzip', 'X-Cache': 'MISS', 'X-Cache-Status': 'NOTCACHED', 'Cache-Control': 'private', 'Content-Disposition': "attachment; filename*=UTF-8''20-12-01-1.xlsx", 'Server': 'ZENEDGE', 'X-Cdn': 'Served-By-Zenedge'}
source: c:\Users\Alan\Documents\GitHub\estatjp\.data\raw\isa\201212\20-12-01-1.xlsx
target: c:\Users\Alan\Documents\GitHub\estatjp\.data\processed\isa\201212\20-12-01-1.csv


## Examine the worksheets and look for pivot tables or column-orientation

There are four known dataformats among the xlsx files
from E-Stat: cross tabulated format (CTF), column-oriented format
(COF), pivot (PVT), and pivot but no cache records. 
Determine format from file.

In [3]:
from estatjp.xlsx.xlsx import get_dataformat

dataformat = get_dataformat(newdata, verbose=True)
print('data format: ' + dataformat)

['20-12-01-1']
title: 20-12-01-1
dimensions: A2:BD209
print_title_cols: $A:$A
print_title_rows: $4:$6
_pivots len: 0
merged_cells len: 22
tableno: 01-1
data format: CTF


## Extract title information

Use `estatjp.xlsx.ctf.parse_title_information()`

In [4]:
from openpyxl import load_workbook
from estatjp.xlsx.ctf import parse_title_information

wb = load_workbook(newdata, data_only=True)
ws = wb.active
titleresults = parse_title_information(ws)
print(titleresults)

{'table_name': '20-12-01-1', 'time_code': 2020001212, 'stats_date': '2020年12月', 'title_population': '在留外国人', 'sub_population': '<NA>', 'cat01_label': '国籍・地域', 'cat02_label': '在留資格', 'cat02_sublabel': '在留目的'}


## Scan the worksheet to isolate tables

Worksheet dimensions are not reliable for determining the columns and rows of the table. Some worksheets have blank columns left and right, and some have blank rows below the table.

The [previous notebook](non-pivot-xlsx-02.ipynb) used tables from 2014 that have merged cell areas that cleanly demarked column headers. Tables from December 2020 do not cleanly use merged cells in the column headers. This notebook will try determining the table dimensions based on the minimum block that contains cells with either string or numerical values like in the previous notebook. It will then determine a subblock containing only numerical values.

For this workbook, the table and its parts are located as shown in the figure below. First the cell range that covers the entire table is determined by counting the maximum number of cells with values among the rows. This cell range becomes `valcellrange`. Next, determine the subrange that contains numerical values. This becomes `numcellrange`. The columns to the left of `numcellrange` become the stub column; those above become the dependent variable cell range `depvarheadrange`, from which the `crowntree` tree structure for parsing the column headers will be derived. 

The row indexes are parsed using a tree structure applied to the stub and stub row range.


<img src="images/TableRanges.gif" width="600">



In [5]:
from openpyxl import load_workbook
from estatjp.xlsx.ctf import identify_table_ranges

ws = wb.active
rangeresults = identify_table_ranges(ws, verbose=False)
print(rangeresults)

{'valrange': <CellRange A4:BD208>, 'numrange': <CellRange B7:BD208>, 'colheadrange': <CellRange A4:BD6>, 'depvarheadrange': <CellRange B4:BD6>, 'stubcellrange': <CellRange A4:A6>, 'stubcolrange': <CellRange A7:A208>}


## Build the stub column tree and create the dictionary of row dictionaries.

Use package [`treelib`](https://treelib.readthedocs.io/en/latest/modules.html).

In [6]:
from estatjp.xlsx.ctf import create_stub_column_dictionary_list

stubrows = create_stub_column_dictionary_list(wsheet=ws, title_result=titleresults, ranges_result=rangeresults)
stubrows

[{'table_name': '20-12-01-1',
  'time_code': 2020001212,
  'stats_date': '2020年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': '総数',
  'cat01lvl0_label': '階級0'},
 {'table_name': '20-12-01-1',
  'time_code': 2020001212,
  'stats_date': '2020年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アジア',
  'cat01lvl0_label': '階級0'},
 {'table_name': '20-12-01-1',
  'time_code': 2020001212,
  'stats_date': '2020年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アフガニスタン',
  'cat01lvl0_label': '階級0'},
 {'table_name': '20-12-01-1',
  'time_code': 2020001212,
  'stats_date': '2020年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl0': 'アラブ首長国連邦',
  'cat01lvl0_label': '階級0'},
 {'table_name': '20-12-01-1',
  'time_code': 2020001212,
  'stats_date': '2020年12月',
  'title_population': '在留外国人',
  'sub_populatio

## Build the crown header tree and create the dictionary of column dictionaries.

Use package [`treelib`](https://treelib.readthedocs.io/en/latest/modules.html).

In [7]:
from estatjp.xlsx.ctf import create_crown_header_dictionary_list

columndicts = create_crown_header_dictionary_list(wsheet=ws, title_result=titleresults, ranges_result=rangeresults)
columndicts

[{'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '総数',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '教授',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '芸術',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '宗教',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '報道',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '高度専門職1号イ',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '高度専門職1号ロ',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '高度専門職1号ハ',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': '高度専門職2号',
  'cat02lvl1': '<NA>',
  'cat02lvl2': '<NA>'},
 {'units': '人',
  'cat02': '在留資格',
  'cat02lvl0': 

## Scan the body to make a list of row dictionaries that will be used to create the `pandas.dataframe`

In [8]:
from estatjp.xlsx.ctf import create_dataframe
import csv

df = create_dataframe(wsheet=ws, 
                    ranges_result=rangeresults,
                    row_leaves=stubrows,
                    column_leaves=columndicts
                    )

df.to_csv(processeddata, index=False, quoting=csv.QUOTE_ALL)
df

,table_name,time_code,stats_date,title_population,sub_population,cat01,cat01lvl0,cat01lvl0_label,units,cat02,cat02lvl0,cat02lvl1,cat02lvl2,value
0,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,総数,階級0,人,在留資格,総数,<NA>,<NA>,2887116
1,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,総数,階級0,人,在留資格,教授,<NA>,<NA>,6647
2,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,総数,階級0,人,在留資格,芸術,<NA>,<NA>,448
3,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,総数,階級0,人,在留資格,宗教,<NA>,<NA>,3772
4,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,総数,階級0,人,在留資格,報道,<NA>,<NA>,215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11105,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,無国籍,階級0,人,在留資格,日本人の配偶者等,日本人の配偶者,<NA>,24
11106,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,無国籍,階級0,人,在留資格,日本人の配偶者等,日本人の子,<NA>,24
11107,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,無国籍,階級0,人,在留資格,永住者の配偶者等,<NA>,<NA>,81
11108,20-12-01-1,2020001212,2020年12月,在留外国人,<NA>,国籍・地域,無国籍,階級0,人,在留資格,定住者,<NA>,<NA>,144
